In [1]:
import os
import sys
from io import BytesIO
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

# set setting django
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "sibansos.settings")

# optional: bypass async restriction (penting untuk Jupyter)
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

# --- 2. Setup Django ---
import django
django.setup()

In [15]:
from django.contrib.auth.models import User
from django.contrib.auth import get_user_model, authenticate

from masyarakat.models import Masyarakat

from accounts.api.serializers import UserSerializer, MasyarakatSerializer

from django.core.files.uploadedfile import SimpleUploadedFile

from django.contrib.auth.hashers import make_password

from core.utils.utils import _simpan, _filter_model_data

usecases

In [3]:
from accounts.usecases.auth import Auth
from accounts.usecases.email import EmailService

In [25]:
password_baru = "rdI!AH3yW3RC"
user = User.objects.get(email="dodiwangkarabi@gmail.com")
user.set_password(password_baru)
user.is_active = True
user.save()

In [26]:

username = user.username
user.is_active

True

In [27]:
user = authenticate(username=username, password=password_baru)
user

<User: thomasberger>

In [13]:
user.check_password(password_baru)

True

In [6]:
uc = Auth()
uc.buat_password_random()

'j2#wEfr5Wtuo'

kirim email

In [5]:
EmailService.send_email(
    subject="Contoh yang dikirim",
    message="ini adalah pesan yang dikirim",
    recipient_list=["dodiwangkarabi@gmail.com"]
)

lupa sandi

In [ ]:
from accounts.api.serializers import ForgotPasswordSerializer, ResetPasswordSerializer

UI

In [ ]:
form_data = {
    "email": "lindsay00@example.net"
}

In [ ]:
serializers = ForgotPasswordSerializer(data=form_data)
if serializers.is_valid():
    data = serializers.validated_data
    try:
        user = User.objects.get(email=data["email"])
    except User.DoesNotExist:
        user = None
else:
    data = serializers.errors


if user:
    uc = Auth()
    url = uc.reset_password(user)

url

In [ ]:
uid="Mzg"
token = "daduu9-fac9d928d6a394be272d7419fc0b62f5"

In [ ]:
# import validationeerror
from rest_framework.exceptions import ValidationError

user = uc.get_user_from_reset_link(uid, token)

if not user:
    raise ValidationError(
        "Link reset password tidak valid atau sudah kedaluwarsa."
    )
    
user

In [ ]:
uc.buka_link(uid="Mzg", token="daduja-0fba8e8f36013c7518448f03830ae1b8")

In [ ]:
uc.get_user_from_uid(uid)

In [ ]:
user2 = uc.get_user_from_uid(url)
user2

In [ ]:
urlsafe_base64_decode(uid).decode()

In [ ]:
user = get_user_from_uid(uid)

if not user:
    raise ValidationError("Link reset password tidak valid.")

In [ ]:
from django.utils.http import urlsafe_base64_encode
from django.utils.encoding import force_bytes

uid = urlsafe_base64_encode(force_bytes(user.pk))

In [ ]:
password_baru = {
    "password1": "baru1234",
    "password2": "baru1234"
}

In [ ]:
serializers = ResetPasswordSerializer(data=password_baru)
if serializers.is_valid():
    data = serializers.validated_data
    try:
        # user = User.objects.get(email=data["email"])
        print("berhasil")
    except User.DoesNotExist:
        user = None
else:
    data = serializers.errors
    
data

In [ ]:
user = User.objects.last()

Auth().verifikasi_akun(user, True)

In [ ]:
user.check_password("contoh1234")

In [ ]:
User.objects.last()

In [ ]:
def registrasi_akun(data: dict):
    
    data_masyarakat = _filter_model_data(Masyarakat, data)
    data_user = _filter_model_data(User, data)
    
    data_user["password"] = make_password(data_user["password"]) # hash password
    data_user["is_active"] = False
    
    masyarakat = _simpan(Masyarakat(**data_masyarakat), data_masyarakat)
    user = _simpan(User(**data_user), data_user)
    
    return user, masyarakat
    

def verifikasi_akun(form_data):
    pass

def login(form_data):
    pass

registrasi

In [ ]:
import random
import numpy as np
from faker import Faker

# random.randint(2, 100)
mylist = np.random.randint(1, 9, size=16).tolist()
random_string = "".join([str(x) for x in mylist])
len(random_string)
random_string

In [ ]:
from io import BytesIO

from PIL import Image
from django.core.files.uploadedfile import SimpleUploadedFile


def create_test_image():
    file = BytesIO()

    image = Image.new("RGB", (100, 100), color="white")
    image.save(file, "PNG")

    file.seek(0)

    return SimpleUploadedFile(
        "test.png",
        file.read(),
        content_type="image/png",
    )

In [ ]:
fake = Faker()
fake.name()
fake.user_name()
fake.random_number(digits=16)


form_data

In [ ]:
form_data = {
    "nik": str(fake.random_number(digits=16)),
    "nama": fake.name(),
    "no_hp": f"{fake.random_number(digits=12)}",
    "foto_ktp": create_test_image(),
    "email": fake.email(),
    "username": fake.user_name(),
    "password": "contoh1234",
    "password_confirm": "contoh1234",
    "ulangi_password": "contoh1234",
}

validasi View

In [ ]:
user_serializer = UserSerializer(data=form_data)
masyarakat_serializer = MasyarakatSerializer(data=form_data)

if user_serializer.is_valid() and masyarakat_serializer.is_valid():
    data = {**user_serializer.validated_data, **masyarakat_serializer.validated_data}
    print("berhasil")
else:
    print("gagal")
    print(user_serializer.errors)
    print(masyarakat_serializer.errors)

In [ ]:
# data cleaned
masyarakat_serializer.validated_data
user_serializer.validated_data

In [ ]:
data

In [ ]:
auth_uc = Auth()
auth_uc.registrasi_akun(data)

In [ ]:
# user = User(
#     username=data["username"],
# )

# user.set_password(data["password"])
# user.save()

verifikasi akun

login

In [ ]:
masyarakat = Masyarakat.objects.all()
for m in masyarakat:
    print(m.nama)

In [ ]:
User = get_user_model()

In [ ]:
User

In [ ]:
users = User.objects.all()
for u in users:
    print(u.username)